In [1]:
import pickle

import numpy as np
import pandas as pd
from google.colab import drive

In [2]:
drive.mount('/content/drive')

folder_path = '/content/drive/My Drive/_Project/Files'
file_path = '/content/drive/My Drive/_Project/Files/dataframes_Yuniov.pkl'

Mounted at /content/drive


In [3]:
with open(file_path, 'rb') as f:
    new_data = pickle.load(f)

calls = new_data['calls']
contacts = new_data['contacts']
spend = new_data['spend']
deals = new_data['deals']

#`1. Посчитать юнит-экономику по продуктам.`

Выбираю максимальное число уникальных лидов

In [4]:
display(calls['CONTACTID'].nunique())
display(contacts['Id'].nunique())
display(deals['Contact Name'].nunique())

15215

18510

17954

```
Conventional designation:
UA (Units Acquired) — Number of unique units/customers acquired
C1 (Conversion Rate) — Ratio of actual customers to potential/customers acquired over total prospects
AOV (Average Order Value) — Average revenue per order or transaction
COGS (Cost of Goods Sold) — Variable costs associated with producing goods or services sold
Revenue — Total income generated from sales
T (Total Months of Study) — Total duration or time period measured (e.g., months spent studying)
APC (Average Purchases per Customer) — Average number of purchases/transactions per customer
CLTV (Customer Lifetime Value) — Total gross profit expected from a typical customer over their lifetime
AC (Advertising Cost) — Total spend or budget allocated to marketing and advertising
CAC (Customer Acquisition Cost) — Cost spent to acquire one customer
CPA (Cost Per Acquisition) — Cost to acquire a single conversion or paying customer
CM (Contribution Margin) — Profit remaining after variable costs and acquisition costs
```

In [5]:
ua = contacts['Id'].nunique()
ac = spend['Spend'].sum()
cpa = ac / ua
cogs = 0

In [6]:
def calculate_metrics(
        deals, product_name, ua, cpa, cogs, ac, boost='None', value=1.0):
    mask = (
        (deals['Product'] == product_name) & (deals['Stage'] == 'Payment Done')
    )

    b = deals.loc[mask, 'Contact Name'].count()
    t = deals.loc[mask, 'Months of study'].sum()
    revenue = deals.loc[mask, 'Initial Amount Paid'].sum()

    c1 = b / ua if ua != 0 else 0
    aov = revenue / t if t != 0 else 0
    apc = t / b if b != 0 else 0

    if boost == 'c1':
        c1 = c1 * value
    elif boost == 'aov':
        aov = aov * value
    elif boost == 'apc':
        apc = apc * value

    cac = cpa / c1 if c1 != 0 else 0
    cltv = (aov - cogs) * apc
    cm = (cltv * c1 - cpa) * ua

    return b, c1, t, revenue, aov, apc, cac, cltv, cm


def calculate_metrics_for_products(
        deals, product_name, ua, cpa, cogs, ac, boost='None', value=1.0):
    results = []
    for product in product_name:
        b, c1, t, revenue, aov, apc, cac, cltv, cm = (
            calculate_metrics(
                deals, product, ua, cpa, cogs, ac, boost=boost, value=value)
        )
        results.append({
            'Product': product,
            'UA': ua,
            'C1': c1,
            'B': b,
            'AOV': aov,
            'COGS': cogs,
            'Revenue': revenue,
            'T': t,
            'APC': apc,
            'CLTV': cltv,
            'AC': ac,
            'CAC': cac,
            'CPA': cpa,
            'CM': cm
        })
    return results

In [7]:
product_name = ['Web Developer', 'Digital Marketing', 'UX/UI Design']

results = calculate_metrics_for_products(deals, product_name, ua, cpa, cogs, ac)

columns_unit = [
    'Product', 'UA', 'C1', 'B', 'AOV', 'COGS', 'Revenue', 'T', 'APC',
    'CLTV', 'AC', 'CAC', 'CPA', 'CM'
]

metrics = pd.DataFrame(results, columns=columns_unit)

metrics['C1'] = metrics['C1'].round(3)
metrics[['AOV', 'APC', 'CLTV', 'AC', 'CAC', 'CPA']] = (
    metrics[['AOV', 'APC', 'CLTV', 'AC', 'CAC', 'CPA']].round(2))

display(metrics)

,Product,UA,C1,B,AOV,COGS,Revenue,T,APC,CLTV,AC,CAC,CPA,CM
0,Web Developer,18510,0.007,136,286.87,0,143150.0,499,3.67,1052.57,149523.45,1099.44,8.08,-6373.45
1,Digital Marketing,18510,0.025,472,183.36,0,529900.0,2890,6.12,1122.67,149523.45,316.79,8.08,380376.55
2,UX/UI Design,18510,0.012,229,235.13,0,275100.0,1170,5.11,1201.31,149523.45,652.94,8.08,125576.55


Выводы:
- Лидер по выручке и положительной марже — Digital Marketing: выручка 529 900, маржинальная прибыль 380 376,55, лучший баланс объема заявок и маржинальности.
- Web Developer приносит убыток (CM: –6 373,45). При этом стоимость привлечения одного клиента (CAC) здесь самая высокая — 1 099,44, что негативно отражается на экономике продукта.
- UX/UI Design приносит существенную прибыль (CM: 125 576,55), удерживает средний уровень заказа (AOV: 235,13) и оптимальное соотношение затрат на привлечение и пожизненной ценности клиента (CLTV).

#`2. Из юнит-экономики определить точки роста бизнеса.`

Для расчета точек роста увеличиваю последоватлельно UA, C1, AOV, APC и уменьшаю CPA (на 5%)

In [8]:
ua_plus_5_percent = int(ua * 1.05)
results_ua = calculate_metrics_for_products(
    deals, product_name, ua_plus_5_percent, cpa, cogs, ac)

cpa_minus_5_percent = cpa * 0.95
results_cpa = calculate_metrics_for_products(
    deals, product_name, ua, cpa_minus_5_percent, cogs, ac)

In [9]:
results_c1 = calculate_metrics_for_products(
    deals, product_name, ua, cpa, cogs, ac, boost='c1', value=1.05)

results_aov = calculate_metrics_for_products(
    deals, product_name, ua, cpa, cogs, ac, boost='aov', value=1.05)

results_apc = calculate_metrics_for_products(
    deals, product_name, ua, cpa, cogs, ac, boost='apc', value=1.05)

In [10]:
variants = {
    'Original': results,
    'UA+5%': results_ua,
    'CPA-5%': results_cpa,
    'C1+5%': results_c1,
    'AOV+5%': results_aov,
    'APC+5%': results_apc,
}

data_cm = pd.DataFrame({
    variant: {r['Product']: r['CM'] for r in results}
    for variant, results in variants.items()
})

data_cm.loc['Total'] = data_cm.sum()

data_cm = data_cm.round(2)

display(data_cm)

,Original,UA+5%,CPA-5%,C1+5%,AOV+5%,APC+5%
Web Developer,-6373.45,-13845.58,1102.72,784.05,784.05,784.05
Digital Marketing,380376.55,372904.42,387852.72,406871.55,406871.55,406871.55
UX/UI Design,125576.55,118104.42,133052.72,139331.55,139331.55,139331.55
Total,499579.65,477163.25,522008.17,546987.15,546987.15,546987.15


Вывод: максимальные точки роста выявлены при увеличении конверсии, среднего чека и среднего числа сделок на одного клиента. Так как значения при изменении С1, AOV и APC одинаковые, но не хватает данных и информации для более детального анализа, я предполагаю, что наилучшей точкой роста будет улучшение метрики конверсии (гипотетически средний чек сложно увеличить - может случится отток клиентов, а APC с учетом специфики бизнеса (длительность курсов 6 и 11 месяц) тоже кажется очень долгоиграющей метрикой).

#`3. Понять дерево метрик для бизнеса.`

```
Дерево метрик

Целевой показатель
- Маржинальная прибыль (CM)

Финансовые метрики
- Оборот (Revenue)

Метрики принятия решений
- Число людей, которые обратились (UA)
- Конверсия (C1)
- Средний чек (AOV)
- Стоимость привлечения UA (CPA)
- Среднее число сделок на одного клиента (APC)

Продуктовые метрики
- Число клиентов, совершивших хотя бы одну оплату (B)
- Количество транзакций (T)
- Средняя пожизненная ценность клиента (CLTV)
- Маркетинговый бюджет (AC)
- Стоимость привлечения клиента (CAC)

Атомные метрики
- Все первоначальные столбцы в датасете
```

#`4. Понять на какую метрику продукта они будут воздействовать и сформировать гипотезы.`

Выдвигаю гипотезы по увеличению конверсии

```
Web Developer
Гипотеза: Если мы проведём бесплатный вебинар “с нуля до мини-проекта”, то
потенциальные клиенты смогут попробовать платформу на практике и увидеть
ценность курса, что повысит конверсию (C1).
```

```
Digital Marketing
Гипотеза: Если предложить короткий бесплатный курс “1 неделя результата”, то
потенциальные клиенты смогут почувствовать эффект обучения до покупки полного
курса, что повысит конверсию (C1).
```

```
UX/UI Design.
Гипотеза: Если добавить мини-задание с обратной связью, потенциальные клиенты
смогут попробовать себя в практике и получить фидбэк, повысит конверсию (C1).
```

#`5. Описать метод проверки гипотез с формулированием условия проведения гипотезы.`

```
Web Developer
    Гипотеза (A/B тест):
    Если потенциальные клиенты примут участие в бесплатном вебинаре “с нуля до
    мини-проекта” (группа B), то конверсия (C1) будет на 3,5 процентных пункта
    выше, чем у пользователей, которые не участвовали в вебинаре (группа A — контроль).

    Метод проверки:
    Группа A: пользователи видят обычную страницу курса без вебинара.
    Группа B: пользователи получают приглашение на бесплатный вебинар.

    Метрика: % пользователей, которые зарегистрировались на платный курс (C1).

    Метод оценки: сравнить конверсию C1 у участников вебинара и у контрольной
    группы (клиенты, не участвовавшие в вебинаре).
```

```
Digital Marketing
    Гипотеза (A/B тест):
    Если пользователи пройдут короткий бесплатный курс “1 неделя результата”
    (группа B), то их конверсия (C1) в платный курс будет на 3,5
    процентных пункта выше, чем у контрольной группы (группа A).

    Метод проверки:
    Группа A: пользователи не получают бесплатный курс.
    Группа B: пользователи получают доступ к бесплатному курсу.

    Метрика: % пользователей, которые затем оформили полный курс.

    Метод оценки: сравнить C1 между пользователями, которые прошли бесплатный
    курс, и контрольной группой.
```

```
UX/UI Design
    Гипотеза (A/B тест):
    Если пользователи выполнят мини-задание с обратной связью (группа B), то
    конверсия (C1) будет на 3,5 процентных пункта выше, чем у группы без
    задания (группа A).

    Метод проверки:
    Группа A: пользователи видят стандартную страницу курса.
    Группа B: пользователи получают мини-задание с обратной связью.

    Метрика: % пользователей, которые оформили платный курс.

    Метод оценки: сравнить C1 между группой с мини-заданием и контрольной
    группой без задания.
```

Проверяем возможность проведения наших тестов за 2 недели с учетом среднего привлечения leads в день

Для этого получаем количество дней, в течение которого к нам поступали звонки

In [11]:
min_time = calls['Call Start Time'].min()
max_time = calls['Call Start Time'].max()

days_active = (max_time - min_time).days

print('Earliest time:', min_time)
print('Latest time:', max_time)
print('Difference in days:', days_active)

Earliest time: 2023-06-30 08:43:00
Latest time: 2024-06-21 15:31:00
Difference in days: 357


Считаем, сколько leads в день в среднем к нам приходит

In [12]:
days_active = 357

daily_users = int(ua / days_active)
daily_users

51

In [13]:
def sample_size_for_course(
    contacts, deals, product_name, x=0.035, days=14, daily_users_round=None
):
    ua = contacts['Id'].nunique()
    b = deals.loc[
        (deals['Product'] == product_name) & (deals['Stage'] == 'Payment Done'),
        'Contact Name'
    ].count()

    p = b / ua
    n = (16 * p * (1 - p)) / (x ** 2)
    n_round = round(n)
    two_groups = n_round * 2

    print(f'Course: {product_name}')
    print(f'Baseline conversion (C1): {p:.3f}')
    print(f'Sample size per group: {round(n):,}')
    print(f'Total sample for two groups: {round(two_groups):,}')

    if daily_users_round:
        daily_users_round = round(daily_users_round)
        days_needed = two_groups / daily_users_round
        print(
            f'Estimated days needed based on daily traffic '
            f'({daily_users_round:,}/day): {round(days_needed):,}')
    print()


for course in product_name:
    sample_size_for_course(
        contacts, deals, course, daily_users_round=daily_users
    )

Course: Web Developer
Baseline conversion (C1): 0.007
Sample size per group: 95
Total sample for two groups: 190
Estimated days needed based on daily traffic (51/day): 4

Course: Digital Marketing
Baseline conversion (C1): 0.025
Sample size per group: 325
Total sample for two groups: 650
Estimated days needed based on daily traffic (51/day): 13

Course: UX/UI Design
Baseline conversion (C1): 0.012
Sample size per group: 160
Total sample for two groups: 320
Estimated days needed based on daily traffic (51/day): 6



Данный расчет произведен с учетом ограничения на проведение тестов не более 14 дней и желаемого увеличения конверсии на 3,5 процентных пунктов (x=0.035)